In [1]:
import os
import GEOparse
import pandas as pd

# Get GEO_ID from parent folder name
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))

gse = GEOparse.get_GEO(geo=GEO_ID, silent=True)

# map sample title to metadata
metadata = {}

for key, value in gse.gsms.items():
    sample_title = value.metadata['title'][0]

    metadata_object = {'sample_title': sample_title}
    for item in value.metadata['characteristics_ch1']:
        k, v = item.split(':')
        metadata_object[k.strip()] = v.strip()

    metadata[key] = metadata_object


metadata_df = pd.DataFrame.from_dict(metadata, orient='index')
metadata_df.index.name = 'samples'

metadata_df

,sample_title,subtype,tnm,Stage,grading,cohort,serpinb3 expression (high_low),cancer_death,month
samples,,,,,,,,,
GSM7026054,100_67161,classical,T3N0MX,IIA,G1,UMD-3,L,1,17.47
GSM7026055,106_62660,classical,T3N0MX,IIA,G2,UMD-3,H,1,25.33
GSM7026056,108_56008,classical,T2N0MX,IB,G2-3,UMD-3,H,1,37.5
GSM7026057,109_57484,unclassified,T3N1MX,IIB,G3,UMD-3,L,1,1.77
GSM7026058,11_19927T,basal,T2N0M0,IB,G3,UMD-1,H,1,14.2
...,...,...,...,...,...,...,...,...,...
GSM7026224,9_18590T,unclassified,T3N1M0,IIB,G2,UMD-1,L,1,21.5
GSM7026225,91_67151,classical,T3N1MX,IIB,G3,UMD-3,L,1,1.93
GSM7026226,92_67191,classical,T3N1MX,IIB,G2,UMD-3,H,1,7.07


In [2]:
# Raw counts
ncbi_data_url = f"https://www.ncbi.nlm.nih.gov/geo/download/?type=rnaseq_counts&acc={GEO_ID}&format=file&file={GEO_ID}_raw_counts_GRCh38.p13_NCBI.tsv.gz"

df_raw_counts: pd.DataFrame = pd.read_csv(ncbi_data_url, sep='\t', index_col=0, compression='gzip')
df_raw_counts = df_raw_counts.T
df_raw_counts.index.name = "samples"
df_raw_counts.columns.name = None

url = "https://www.ncbi.nlm.nih.gov/geo/download/?format=file&type=rnaseq_counts&file=Human.GRCh38.p13.annot.tsv.gz"
df_gene_annotations = pd.read_csv(
    url,
    sep="\t",
    compression="gzip"
)
gene_mapping = (
    df_gene_annotations[["GeneID", "Symbol"]]
    .dropna(subset=["GeneID", "Symbol"])
    .set_index("GeneID")["Symbol"]
    .to_dict()
)
# Keep only columns that have a mapping
mappable = [c for c in df_raw_counts.columns if c in gene_mapping]
df_raw_counts = df_raw_counts[mappable].rename(columns=gene_mapping)
# df_raw_counts = df_raw_counts.rename(columns=gene_mapping)

data_file_name = f'{GEO_ID}_raw_counts_NCBI.csv'
df_combined = metadata_df.join(df_raw_counts, how='inner')




df_combined.reset_index()
df_combined.to_csv(data_file_name, index=False)

df_combined

/var/folders/h1/fnfx16m50jvd8ddpcs93g9mw0000gn/T/ipykernel_6962/3329876131.py:10: DtypeWarning: Columns (8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df_gene_annotations = pd.read_csv(


,sample_title,subtype,tnm,Stage,grading,cohort,serpinb3 expression (high_low),cancer_death,month,DDX11L1,...,ND4,TRNH,TRNS2,TRNL2,ND5,ND6,TRNE,CYTB,TRNT,TRNP
samples,,,,,,,,,,,,,,,,,,,,,
GSM7026054,100_67161,classical,T3N0MX,IIA,G1,UMD-3,L,1,17.47,2,...,8373,11,9,8,5714,514,230,6799,19,280
GSM7026055,106_62660,classical,T3N0MX,IIA,G2,UMD-3,H,1,25.33,17,...,13307,31,32,33,6501,686,304,5344,47,591
GSM7026056,108_56008,classical,T2N0MX,IB,G2-3,UMD-3,H,1,37.5,5,...,7111,10,10,9,5630,469,221,4530,37,284
GSM7026057,109_57484,unclassified,T3N1MX,IIB,G3,UMD-3,L,1,1.77,4,...,8101,31,28,25,5176,357,224,6712,66,259
GSM7026058,11_19927T,basal,T2N0M0,IB,G3,UMD-1,H,1,14.2,2,...,4104,10,12,10,2122,222,88,3174,47,152
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GSM7026223,87_64570,basal,T3N0MX,IIA,G3,UMD-3,H,1,28.2,2,...,6248,10,9,10,3664,348,179,5862,17,154
GSM7026224,9_18590T,unclassified,T3N1M0,IIB,G2,UMD-1,L,1,21.5,6,...,7273,6,10,10,4256,413,150,5872,12,193
GSM7026226,92_67191,classical,T3N1MX,IIB,G2,UMD-3,H,1,7.07,4,...,3215,19,17,14,2326,254,72,1733,36,131


# Generate description

In [3]:
import json
import os
import sys
from pathlib import Path

project_root = Path().resolve().parent.parent  # adjust based on notebook depth
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from geo.scripts.generate_descriptions import generate_description_from_geo

description_file_path = 'description.json'


if not os.path.exists(description_file_path):
    with open(description_file_path, 'w') as f:
        json.dump({}, f)

with open(description_file_path, 'r') as f:
    desc_data = json.load(f)

if not desc_data:
    desc_data = generate_description_from_geo(GEO_ID)
    print("Description generated")

desc_data['data_summary'] = {
    'samples': metadata_df.shape[0],
    'clinical_features': metadata_df.shape[1],
    'genes': df_raw_counts.shape[1]
}
print("Description updated")

desc_data['data_file_names'] = [data_file_name]

with open(description_file_path, "w") as f:
    json.dump(desc_data, f, indent=4)

Description updated
